# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [9]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled',
 '/mistralai/Voxtral-4B-TTS-2603',
 '/HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive',
 '/CohereLabs/cohere-transcribe-03-2026',
 '/nvidia/Nemotron-Cascade-2-30B-A3B',
 '/models',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/victor/dlss-5-anything',
 '/spaces/deddytoyota/Free-Unlimited-Google-Veo-3',
 '/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast',
 '/spaces/FrameAI4687/Omni-Video-Factory',
 '/spaces',
 '/datasets/OpenMOSS-Team/OmniAction',
 '/datasets/open-index/hacker-news',
 '/datasets/th1nhng0/vietnamese-legal-documents',
 '/datasets/OpenMOSS-Team/OmniAction-LIBERO',
 '/datasets/ServiceNow-AI/eva',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pric

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/login
/join
/spaces
/models
/Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
/mistralai/Voxtral-4B-TTS-2603
/HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
/CohereLabs/cohere-transcribe-03-2026
/nvidia/Nemotron-Cascade-2-30B-A3B
/models
/spaces/r3gm/wan2-2-fp8da-aoti-preview
/spaces/victor/dlss-5-anything
/spaces/deddytoyota/Free-Unlimited-Google-Veo-3
/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast
/spaces/FrameAI4687/Omni-Video-Factory
/spaces
/datasets/OpenMOSS-Team/OmniAction
/datasets/open-index/hacker-news
/datasets/th1nhng0/vietnamese-legal-documents
/datasets/OpenMOSS-Team/OmniAction-LIBERO
/datasets/Se

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'API endpoints page', 'url': 'https://endpoints.huggingface.co'}]}

In [14]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 19 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
4 days ago
•
253k
•
1.49k
mistralai/Voxtral-4B-TTS-2603
Updated
about 21 hours ago
•
1.8k
•
353
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
18 days ago
•
479k
•
1.02k
CohereLabs/cohere-transcribe-03-2026
Updated
about 10 hours ago
•
12.1k
•
306
nvidia/Nemotron-Cascade-2-30B-A3B
Updated
3 days ago
•
69.6k
•
349
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.63k
Wan2.2 14B Preview
🐌
1.63k
generate a video from an image with a text prompt
Running
on
Zero
Featured
272
DLSS 5 Anything
🎮
272
Turn any image into a DLSS 5 meme (using FLUX.2-klein-9b-kv

In [25]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n4 days ago\n•\n253k\n•\n1.49k\nmistralai/Voxtral-4B-TTS-2603\nUpdated\nabout 21 hours ago\n•\n1.8k\n•\n353\nHauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\n18 days ago\n•\n479k\n•\n1.02k\nCohereLabs/cohere-transcribe-03-2026\nUpdated\nabout 10 hours ago\n•\n12.1k\n•\n306\nnvidia/Nemotron-Cascade-2-30B-A3

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. As a leading collaboration platform, Hugging Face empowers data scientists, ML engineers, researchers, and developers worldwide to create, explore, discover, and share models, datasets, and applications openly. The platform supports multiple data modalities including text, image, video, audio, and 3D, fostering innovation across various AI frontiers.

With over 2 million machine learning models, half a million datasets, and a thriving ecosystem of AI applications, Hugging Face stands at the heart of the AI revolution. Its open-source libraries and tools are some of the most popular globally, enabling rapid experimentation and deployment of AI solutions.

---

## Platform Highlights

- **Model Hub:** Access and contribute to a repository with 2M+ state-of-the-art machine learning models.
- **Datasets:** Browse and share 500k+ datasets across multiple domains.
- **Spaces:** Host and run machine learning applications powered by popular frameworks.
- **Collaboration:** Host unlimited public projects, engage with an active community, and accelerate your ML workflows.
- **Open Source Ecosystem:** Benefit from a rich stack of free and open-source libraries and tools designed to make ML development fast and seamless.

---

## Enterprise Solutions

Hugging Face offers tailored plans to help teams and organizations scale AI initiatives securely and efficiently:

- **Team Plan** starting at $20/user/month
- **Enterprise Plan** with flexible contract terms and dedicated support

Key Enterprise Features:

- Enterprise-grade security with Single Sign-On (SSO)
- Granular access control and audit logging
- Data residency options and private storage expansion
- Centralized token management and resource group allocation
- Advanced compute scaling including ZeroGPU quota boosts
- Analytics dashboards for usage monitoring and cost tracking
- Private datasets viewer to facilitate confidential collaboration

---

## Company Culture

Hugging Face is a passionate, inclusive, and ethical AI community focused on openness and collaboration. The company philosophy centers around:

- **Building an open and ethical AI future together**
- Empowering the next generation of machine learning engineers and scientists
- Encouraging knowledge sharing, experimentation, and creativity
- Maintaining transparency and accessibility in AI development
- Cultivating a diverse, global community of AI practitioners

---

## For Prospective Customers

Join thousands of innovative companies and research organizations leveraging Hugging Face to accelerate AI development. Whether you need to prototype ideas, build production models, or collaborate across teams, Hugging Face provides the tools and infrastructure to:

- Speed up ML pipelines
- Access cutting-edge models and datasets
- Ensure data security and governance at scale
- Gain insights with usage analytics
- Deploy AI applications reliably and efficiently

---

## Careers at Hugging Face

Hugging Face seeks talented individuals who are passionate about AI and open source to join their growing teams. Working at Hugging Face offers the opportunity to:

- Collaborate with top-tier scientists and engineers
- Contribute to projects that impact millions worldwide
- Work in a culture valuing openness, curiosity, and impact
- Innovate at the frontier of natural language processing, computer vision, audio processing, and more

Explore open roles on their website and become part of the AI community shaping the future.

---

## Connect With Hugging Face

- Website: https://huggingface.co 
- GitHub | Twitter | LinkedIn | Discord  
- Engage with over a million members sharing ideas, knowledge, and breakthroughs in machine learning every day.

---

**Hugging Face — The AI Community Building the Future**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face: The AI Community Building the Future

---

## Who We Are

Hugging Face is the leading collaboration platform dedicated to the machine learning (ML) community. We empower ML engineers, scientists, and enthusiasts worldwide to share, explore, and develop state-of-the-art open-source models, datasets, and applications. Our fast-growing global community is committed to building an open, ethical, and inclusive AI future together.

---

## Our Platform

- **Models & Datasets:** Access and contribute to over 2 million models and 500,000 datasets across a variety of modalities such as text, image, video, audio, and even 3D.
- **Spaces:** Host and launch ML-powered web applications easily with a rich environment to run demos and prototypes.
- **Buckets:** Store and manage large datasets and machine learning assets securely.
- **Open Source Stack:** Move faster and innovate more effectively using our comprehensive and freely available ML tools and libraries.

Our platform is designed as a central hub to create, discover, and collaborate on machine learning projects at scale, facilitating wide accessibility for all skill levels—from beginners building portfolios to experts driving advanced research.

---

## Our Community and Customers

Hugging Face supports a diverse and active community comprising:

- Machine learning researchers and data scientists pushing the boundaries of AI research.
- Developers building cutting-edge applications and AI solutions.
- Enterprises integrating ML models to transform their business operations.
- Educators and students learning and sharing knowledge in AI and ML.

Join millions of users worldwide who leverage Hugging Face for innovation — from small startups to global corporations.

---

## Company Culture

- **Collaboration:** We foster an open and supportive environment where knowledge and resources are freely shared.
- **Ethical AI:** Commitment to building AI responsibly with transparency and inclusivity at the core.
- **Community-driven:** Continuous growth fueled by contributions from a passionate and engaged AI community.
- **Innovation:** Encouraging experimentation and creativity to accelerate the future of machine learning.
- **Learning & Growth:** Providing tools and opportunities for personal and professional development in AI.

---

## Careers at Hugging Face

Are you passionate about AI and machine learning? Hugging Face offers exciting career opportunities to:

- Develop infrastructure and tools for scalable ML systems.
- Lead initiatives in research, engineering, product development, and community engagement.
- Work alongside world-class experts in a mission-driven startup environment.
- Contribute to open-source projects that impact millions globally.

Join us in building the future of artificial intelligence. Be part of a dynamic, inclusive, and forward-thinking team.

---

## Explore More

- Visit [huggingface.co](https://huggingface.co) to browse models, datasets, and AI applications.
- Engage with our community through forums and collaborative spaces.
- Access documentation and tutorials to accelerate your ML projects.
- Learn about enterprise solutions and pricing for business needs.

---

**Hugging Face**  
Building the future, together.  
The Home of Machine Learning Collaboration.

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Welcome to Hugging Face: The AI Community Building the Future 🤗🚀

---

### Who Are We?

Hugging Face is not just a cute name — we're the *collaboration platform* where the global machine learning community comes alive! Think of us as the buzzing hive where ML engineers, scientists, developers, and AI enthusiasts share models, datasets, and build applications that push the boundaries of AI.

With **2 million+ models**, **500k+ datasets**, and **1 million+ applications**, we’re basically the Netflix and chill zone for everything Machine Learning. Whether you’re into text, image, video, audio, or even 3D — we’ve got you covered.

---

### What’s In Our AI Magic Toolbox? 🧙‍♂️✨

- **Models Galore:** From cutting-edge language models to speech synthesis and computer vision, our community constantly updates thousands of models. Trending now? Try “Jackrong/Qwen3.5-27B” or “mistralai/Voxtral-4B-TTS-2603” for the latest AI wizardry.
- **Spaces:** Launch your own AI apps and play with zero setup. From fun meme generators like DLSS 5 Anything 🎮 to serious multi-modal video factories 🏆 — there’s a space for every AI dream.
- **Datasets:** Free and open! Explore datasets across languages, legal docs, or even hack your newsfeed. Fresh data = fresh AI.

---

### Why Hugging Face? Because We’re More Than Just Code 🤝

- **Open & Ethical:** We’re building an AI future that’s open to all and ethically sound. Share your work, learn from others, and together let’s make AI for good.
- **Community Spirit:** Join a fast-growing army of ML geeks who love to collaborate and push AI frontiers.
- **Fast-Paced Innovation:** With our open-source stack, you skip the boring parts and dive straight into building and experimenting.
- **Multi-Modal Fun:** Text, images, video, audio, 3D — because why limit your AI fancy?

---

### Careers: Join the AI Revolution! 🚀

Looking for a place where your code meets community and impact? Hugging Face is hiring curious minds who want to:

- Build open-source tools loved by tens of thousands daily.
- Collaborate with world-class researchers and developers.
- Champion ethical AI in everything they do.

Whether you're an engineer, data scientist, or AI enthusiast, here’s your chance to be part of the AI heartbeat.

---

### Customers & Collaborators: Who’s Hugging Who?

Our platform is the go-to for researchers, startups, AI teams at leading companies (hello Nvidia, CohereLabs!), and anyone who wants to build or use top-tier AI models without starting from scratch. Big or small, if AI’s your game, we’re your home.

---

### Brand Vibes: Sunny Yellow, Friendly Faces, and Friendly Codes 🌞🤖

Our colors scream innovation and positivity (#FFD21E, #FF9D00), just like our vibe — warm, approachable, and extremely smart. Our mascot? Well, it’s hugging the AI future — literally!

---

### Ready to Hug the Future?

- Visit [huggingface.co](https://huggingface.co) to explore models, datasets, and AI apps.
- Join the community, share your genius, or dive into your next AI project.
- Because at Hugging Face, we’re building the future — one model, one dataset, one hello at a time.

---

*Join us and let’s create AI magic together. Warning: hugging may cause excessive creativity and community spirit!* 🤗✨

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>